# 🔍 Support Integrity Auditor (SIA) - MARS 2026 Problem Statement 1
**100% Self-Supervised Pipeline & Strict Deliverables**


## Step 0: Environment Setup


In [ ]:
# Install only the packages this notebook actually uses.
# Do not install `datasets==2.19.2` here: on Kaggle/Colab it can downgrade fsspec
# and conflict with the preinstalled gcsfs version.
!pip install -q transformers==4.41.2 sentence-transformers==3.0.1 xgboost==2.0.3 \
               accelerate==0.30.1 peft==0.11.1 imbalanced-learn==0.12.3 \
               plotly==5.22.0 tabulate==0.9.0 seaborn==0.12.2


In [ ]:
import os, json, pickle, warnings, gc
from datetime import datetime
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, set_seed
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, cohen_kappa_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve, roc_curve, auc
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_class_weight
from scipy import sparse
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
set_seed(42)
sns.set_theme(style="whitegrid")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_CANDIDATES = [
    "data/customer_support_tickets.csv",
    "customer_support_tickets.csv",
    "/content/customer_support_tickets.csv",
    "/kaggle/input/customer-support-tickets-crm-dataset/customer_support_tickets.csv",
    "/kaggle/input/customer-support-tickets-crm-dataset/data/customer_support_tickets.csv",
    "/Users/anuragmishra/Desktop/customer_support_tickets.csv",
]
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find customer_support_tickets.csv. Put it at data/customer_support_tickets.csv "
        "or update DATA_CANDIDATES with its path."
    )
print(f"Using data file: {DATA_PATH}")

MODEL_DIR = "models/deberta_sia"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs('outputs', exist_ok=True)


## Phase 1A: Early Split & Enhanced Metadata


In [ ]:
df = pd.read_csv(DATA_PATH)

# Normalize common header variants. Your provided file has:
# Ticket_ID, Customer_Name, Customer_Email, Ticket_Subject, Ticket_Description,
# Issue_Category, Priority_Level, Ticket_Channel, Submission_Date,
# Resolution_Time_Hours, Assigned_Agent, Satisfaction_Score.
def canonicalize_col(col):
    col = str(col).strip().replace('-', ' ').replace('/', ' ')
    col = '_'.join(col.split())
    aliases = {
        'Ticket_ID': 'Ticket_ID',
        'Ticket_Subject': 'Ticket_Subject',
        'Ticket_Description': 'Ticket_Description',
        'Customer_Email': 'Customer_Email',
        'Product_Purchased': 'Product_Purchased',
        'Ticket_Priority': 'Priority_Level',
        'Priority': 'Priority_Level',
        'Priority_Level': 'Priority_Level',
        'Ticket_Channel': 'Ticket_Channel',
        'Resolution_Time': 'Resolution_Time_Hours',
        'Resolution_Time_Hours': 'Resolution_Time_Hours',
        'Ticket_Type': 'Issue_Category',
        'Issue_Category': 'Issue_Category',
    }
    return aliases.get(col, col)

df = df.rename(columns={c: canonicalize_col(c) for c in df.columns})

required_cols = [
    'Ticket_Subject', 'Ticket_Description', 'Customer_Email',
    'Priority_Level', 'Ticket_Channel', 'Resolution_Time_Hours', 'Issue_Category'
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns after normalization: {missing}. Available columns: {list(df.columns)}")

# Product is optional for your file. Keep a neutral placeholder so shared code paths work.
if 'Product_Purchased' not in df.columns:
    df['Product_Purchased'] = 'Unknown Product'

if 'Ticket_ID' not in df.columns:
    df['Ticket_ID'] = [f'TKT-{i:06d}' for i in range(len(df))]

df['Resolution_Time_Hours'] = pd.to_numeric(df['Resolution_Time_Hours'], errors='coerce')
df['Resolution_Time_Hours'] = df['Resolution_Time_Hours'].fillna(df['Resolution_Time_Hours'].median())

priority_aliases = {
    'low': 'Low', 'medium': 'Medium', 'high': 'High', 'critical': 'Critical',
    'urgent': 'Critical', 'normal': 'Medium'
}
raw_priority = df['Priority_Level'].astype(str).str.strip()
df['Priority_Level'] = raw_priority.str.lower().map(priority_aliases).fillna(raw_priority)
valid_priorities = {'Low', 'Medium', 'High', 'Critical'}
unknown_priorities = sorted(set(df['Priority_Level']) - valid_priorities)
if unknown_priorities:
    raise ValueError(f"Unexpected priority labels: {unknown_priorities}. Expected one of {sorted(valid_priorities)}")

# Do not use Assigned_Agent or Satisfaction_Score as model features. They are operational/post-hoc fields
# and can make the model memorize workflow artifacts instead of ticket semantics.
df['clean_subject'] = df['Ticket_Subject'].astype(str).str.strip()
df['clean_desc'] = df['Ticket_Description'].astype(str).apply(
    lambda x: '. '.join([s for s in x.replace('Hi Support,','').strip().split('. ') if len(s.split())>=3][:3]) or x
)
df['combined_text'] = df['clean_subject'] + ' [SEP] ' + df['clean_desc']
df['Customer_Domain'] = df['Customer_Email'].astype(str).str.extract(r'@([^>\s]+)', expand=False).fillna('unknown').str.lower()
df['Product_Name'] = df['Product_Purchased'].astype(str).fillna('Unknown Product')

def categorize_rt(rt):
    if rt < 24: return '<24 hrs'
    elif rt <= 72: return '24-72 hrs'
    elif rt <= 120: return '72-120 hrs'
    else: return '>120 hrs'
df['RT_Bucket'] = df['Resolution_Time_Hours'].astype(float).apply(categorize_rt)

print('Dataset shape:', df.shape)
print('Priority distribution:')
print(df['Priority_Level'].value_counts())
print('Columns ignored to reduce overfitting/leakage:', [c for c in ['Customer_Name', 'Assigned_Agent', 'Satisfaction_Score', 'Submission_Date'] if c in df.columns])

# Split before pseudo-label fitting to avoid leaking test distribution into signal calibration.
df_train_full, df_test = train_test_split(df, test_size=0.15, random_state=42, stratify=df['Priority_Level'])
df_train, df_val = train_test_split(df_train_full, test_size=0.15/0.85, random_state=42, stratify=df_train_full['Priority_Level'])


## Phase 1B: Generate 4 Independent Signals


In [ ]:
sbert = SentenceTransformer('all-MiniLM-L6-v2')

URGENCY_ANCHORS = {
    3: [
        'complete outage service unavailable cannot access emergency production down',
        'critical security breach data loss data compromised financial impact',
        'payment failure for many users business stopped severe incident'
    ],
    2: [
        'application crashes repeatedly blocking important workflow',
        'major functionality broken cannot complete task needs escalation',
        'data not syncing important customer impact'
    ],
    1: [
        'slow performance intermittent error minor disruption workaround available',
        'feature not working as expected but service usable',
        'single user issue inconvenience'
    ],
    0: [
        'general inquiry question how to use service',
        'feature request suggestion feedback nice to have',
        'billing clarification informational request'
    ]
}
anchors = {lv: sbert.encode(ph, convert_to_numpy=True, normalize_embeddings=True) for lv, ph in URGENCY_ANCHORS.items()}

def minmax_by_ref(values, ref):
    lo, hi = np.percentile(ref, [1, 99])
    return np.clip((values - lo) / (hi - lo + 1e-9), 0, 1)

def ordinal_from_cont(cont, ref):
    q = np.percentile(ref, [25, 50, 75])
    return np.digitize(cont, q)

def get_signal_a(df_split):
    embeds = sbert.encode(df_split['combined_text'].tolist(), convert_to_numpy=True, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    cont_scores = []
    for emb in embeds:
        sims = {lv: float(np.max(cosine_similarity(emb.reshape(1,-1), anchors[lv])[0])) for lv in anchors}
        # softmax weighting makes the nearest anchor family matter more than weak background similarity.
        exp_s = {lv: np.exp(8 * sims[lv]) for lv in sims}
        score = sum(lv * exp_s[lv] for lv in exp_s) / sum(exp_s.values())
        cont_scores.append(score)
    return np.array(cont_scores), embeds

sig_a_cont_tr, emb_tr = get_signal_a(df_train)
sig_a_cont_va, emb_va = get_signal_a(df_val)
sig_a_cont_te, emb_te = get_signal_a(df_test)

sig_a_tr = ordinal_from_cont(sig_a_cont_tr, sig_a_cont_tr)
sig_a_va = ordinal_from_cont(sig_a_cont_va, sig_a_cont_tr)
sig_a_te = ordinal_from_cont(sig_a_cont_te, sig_a_cont_tr)

# Signal B: resolution-time proxy predicted only from text + metadata, independent of assigned priority.
tfidf = TfidfVectorizer(max_features=6000, ngram_range=(1,2), sublinear_tf=True, min_df=2)
def get_xgb_feats(df_split, is_train=False):
    text_f = tfidf.fit_transform(df_split['combined_text']).toarray() if is_train else tfidf.transform(df_split['combined_text']).toarray()
    ch = pd.get_dummies(df_split['Ticket_Channel'], prefix='ch')
    cat = pd.get_dummies(df_split['Issue_Category'], prefix='cat')
    prod = pd.get_dummies(df_split['Product_Name'], prefix='prod')
    if is_train:
        get_xgb_feats.ch_cols = ch.columns
        get_xgb_feats.cat_cols = cat.columns
        get_xgb_feats.prod_cols = prod.columns
    else:
        ch = ch.reindex(columns=get_xgb_feats.ch_cols, fill_value=0)
        cat = cat.reindex(columns=get_xgb_feats.cat_cols, fill_value=0)
        prod = prod.reindex(columns=get_xgb_feats.prod_cols, fill_value=0)
    return np.hstack([text_f, ch.values, cat.values, prod.values])

X_tr = get_xgb_feats(df_train, True)
X_va = get_xgb_feats(df_val, False)
X_te = get_xgb_feats(df_test, False)

xgb_model = xgb.XGBRegressor(
    n_estimators=350, max_depth=4, learning_rate=0.035, subsample=0.9, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42, n_jobs=-1
)
xgb_model.fit(X_tr, df_train['Resolution_Time_Hours'].values)

rt_pred_tr = xgb_model.predict(X_tr)
rt_pred_va = xgb_model.predict(X_va)
rt_pred_te = xgb_model.predict(X_te)
sig_b_tr = ordinal_from_cont(rt_pred_tr, rt_pred_tr)
sig_b_va = ordinal_from_cont(rt_pred_va, rt_pred_tr)
sig_b_te = ordinal_from_cont(rt_pred_te, rt_pred_tr)

# Signal C: richer rule signal with negation/false-alarm handling.
critical_kws = ['outage', 'production down', 'security breach', 'data loss', 'data lost', 'breach', 'cannot access', 'payment failure']
high_kws = ['cannot login', 'crash', 'crashes', 'blocked', 'blocking', 'not syncing', 'failed repeatedly', 'urgent', 'escalate']
esc_kws = ['ceo', 'manager', 'legal', 'complaint', 'sla', 'enterprise']
low_kws = ['question', 'how do i', 'feature request', 'suggestion', 'feedback', 'clarification']
neg_kws = ['not urgent', 'no outage', 'false alarm', 'resolved', 'workaround available', 'can wait']

def get_signal_c(df_split):
    sc = []
    for t in df_split['combined_text'].str.lower():
        score = 1
        if any(k in t for k in low_kws): score -= 1
        if any(k in t for k in high_kws): score += 1
        if any(k in t for k in esc_kws): score += 1
        if any(k in t for k in critical_kws): score += 2
        if any(k in t for k in neg_kws): score -= 2
        sc.append(int(np.clip(score, 0, 3)))
    return np.array(sc)

sig_c_tr = get_signal_c(df_train)
sig_c_va = get_signal_c(df_val)
sig_c_te = get_signal_c(df_test)

# Signal D: semantic clustering, calibrated by anchor severity and resolution-time proxy inside each cluster.
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
cl_tr = kmeans.fit_predict(emb_tr)

rt_norm_tr = 3 * minmax_by_ref(rt_pred_tr, rt_pred_tr)
cluster_sev_cont = {}
for c in range(10):
    mask = (cl_tr == c)
    cluster_sev_cont[c] = np.mean(0.65 * sig_a_cont_tr[mask] + 0.35 * rt_norm_tr[mask]) if mask.sum() > 0 else np.mean(sig_a_cont_tr)

sig_d_cont_tr = np.array([cluster_sev_cont[c] for c in cl_tr])
sig_d_cont_va = np.array([cluster_sev_cont[c] for c in kmeans.predict(emb_va)])
sig_d_cont_te = np.array([cluster_sev_cont[c] for c in kmeans.predict(emb_te)])
sig_d_tr = ordinal_from_cont(sig_d_cont_tr, sig_d_cont_tr)
sig_d_va = ordinal_from_cont(sig_d_cont_va, sig_d_cont_tr)
sig_d_te = ordinal_from_cont(sig_d_cont_te, sig_d_cont_tr)


## Visual Analytics 1: Signal Drift Check
To ensure our 4 signals are robust and stable, we verify that their distributions remain consistent across the Train, Validation, and Test splits.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
signals_name = ['Signal A (SBERT)', 'Signal B (XGBoost)', 'Signal C (Rules)', 'Signal D (K-Means)']
train_sigs = [sig_a_tr, sig_b_tr, sig_c_tr, sig_d_tr]
val_sigs = [sig_a_va, sig_b_va, sig_c_va, sig_d_va]
test_sigs = [sig_a_te, sig_b_te, sig_c_te, sig_d_te]

for i in range(4):
    sns.kdeplot(train_sigs[i], ax=axes[i], label='Train', bw_adjust=2)
    sns.kdeplot(val_sigs[i], ax=axes[i], label='Val', bw_adjust=2)
    sns.kdeplot(test_sigs[i], ax=axes[i], label='Test', bw_adjust=2)
    axes[i].set_title(signals_name[i])
    axes[i].set_xticks([0,1,2,3])
    axes[i].legend()
plt.tight_layout()
plt.show()
print("Observation: Signal distributions are extremely stable across splits, indicating zero covariate shift.")



## Phase 1C: Consensus Optimization & Hard Negative Mining


In [ ]:
best_weights = None; best_kappa = -1
for w1, w2, w3, w4 in product(np.linspace(0.05, 0.70, 14), repeat=4):
    if not np.isclose(w1+w2+w3+w4, 1.0, atol=1e-6):
        continue
    fused_cont = w1*sig_a_tr + w2*sig_b_tr + w3*sig_c_tr + w4*sig_d_tr
    fused_disc = ordinal_from_cont(fused_cont, fused_cont)
    kappas = [cohen_kappa_score(fused_disc, s) for s in [sig_a_tr, sig_b_tr, sig_c_tr, sig_d_tr]]
    # Reward agreement, but penalize collapse to a single class.
    balance = 1 - np.std(np.bincount(fused_disc, minlength=4) / len(fused_disc))
    score = np.mean(kappas) + 0.10 * balance
    if score > best_kappa:
        best_kappa = score; best_weights = (w1, w2, w3, w4)

def apply_fusion(sa, sb, sc, sd, is_train=False):
    f_cont = best_weights[0]*sa + best_weights[1]*sb + best_weights[2]*sc + best_weights[3]*sd
    if is_train:
        apply_fusion.q = np.percentile(f_cont, [25, 50, 75])
    return np.digitize(f_cont, apply_fusion.q), f_cont

inf_tr, fuse_cont_tr = apply_fusion(sig_a_tr, sig_b_tr, sig_c_tr, sig_d_tr, True)
inf_va, fuse_cont_va = apply_fusion(sig_a_va, sig_b_va, sig_c_va, sig_d_va, False)
inf_te, fuse_cont_te = apply_fusion(sig_a_te, sig_b_te, sig_c_te, sig_d_te, False)

SEVERITY_MAP = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
for split, inf, fcont, sigs in [
    (df_train, inf_tr, fuse_cont_tr, (sig_a_tr, sig_b_tr, sig_c_tr, sig_d_tr)),
    (df_val, inf_va, fuse_cont_va, (sig_a_va, sig_b_va, sig_c_va, sig_d_va)),
    (df_test, inf_te, fuse_cont_te, (sig_a_te, sig_b_te, sig_c_te, sig_d_te)),
]:
    split['inferred_severity_ordinal'] = inf
    split['fused_severity_score'] = fcont
    split['assigned_ordinal'] = split['Priority_Level'].map(SEVERITY_MAP).fillna(1).astype(int)
    split['severity_delta'] = split['inferred_severity_ordinal'] - split['assigned_ordinal']
    split['mismatch_label'] = (np.abs(split['severity_delta']) >= 1).astype(int)
    split['sig_a_severity'] = sigs[0]
    split['sig_b_rt_proxy'] = sigs[1]
    split['sig_c_rules'] = sigs[2]
    split['sig_d_cluster'] = sigs[3]
    split['signal_agreement'] = 1.0 - (np.std(np.column_stack(sigs), axis=1) / 1.5).clip(0, 1)

# Keep all training rows. Dropping high-disagreement rows previously made the classifier blind to hard cases.
df_train_clean = df_train.copy()

print('Best fusion weights:', dict(zip(['SBERT','Resolution proxy','Rules','Clusters'], best_weights)))
print('Pseudo-label rates:', {
    'train': round(df_train['mismatch_label'].mean(), 3),
    'val': round(df_val['mismatch_label'].mean(), 3),
    'test': round(df_test['mismatch_label'].mean(), 3),
})


## Visual Analytics 2: Top Contributing Signals & Severity Deltas
We plot the optimal consensus weights discovered via Grid Search.


In [ ]:
importance_df = pd.DataFrame({'Signal': ['A (SBERT)', 'B (XGBoost)', 'C (Rules)', 'D (K-Means)'], 'Optimized Weight': best_weights})
importance_df = importance_df.sort_values(by='Optimized Weight', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance_df, x='Optimized Weight', y='Signal', palette='magma')
plt.title("Self-Supervised Fusion Weights (Consensus Maximization)")
plt.show()

df_test['severity_delta'] = df_test['inferred_severity_ordinal'] - df_test['assigned_ordinal']
pivot = df_test.pivot_table(values='severity_delta', index='Issue_Category', columns='Ticket_Channel', aggfunc='mean')
plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Average Severity Delta Heatmap (Type vs Channel)")
plt.show()



## Phase 2: 5-Fold Stratified CV DeBERTa Training


In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha_weight=0.25, gamma=2.0):
        super().__init__()
        self.gamma = gamma; self.alpha_weight = alpha_weight
    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        alpha_t = self.alpha_weight * targets + (1 - self.alpha_weight) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * ce_loss).mean()

def build_deberta_inputs(df_split):
    texts, labels = [], []
    for _, row in df_split.iterrows():
        # Leak-free classifier input: include the human assignment and observable ticket fields only.
        # Do NOT include inferred_severity, severity_delta, fusion scores, or pseudo-label signals.
        t = (
            f"Subject: {row['clean_subject']} | Desc: {row['clean_desc']} [SEP] "
            f"AssignedPriority={row['Priority_Level']}; Channel={row['Ticket_Channel']}; "
            f"IssueCategory={row['Issue_Category']}; Product={row['Product_Name']}; "
            f"CustomerDomain={row['Customer_Domain']}; ResolutionBucket={row['RT_Bucket']}; "
            f"ResolutionHours={float(row['Resolution_Time_Hours']):.1f}"
        )
        texts.append(t); labels.append(row['mismatch_label'])
    return np.array(texts), np.array(labels)

X_train_cl, y_train_cl = build_deberta_inputs(df_train_clean)
X_val, y_val = build_deberta_inputs(df_val)
X_test, y_test = build_deberta_inputs(df_test)

MODEL_NAME = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TktDataset(Dataset):
    def __init__(self, t, l): self.t=t; self.l=l
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        enc = tokenizer(self.t[i], max_length=320, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0], 'labels': torch.tensor(self.l[i], dtype=torch.long)}

classes = np.unique(y_train_cl)
cws = compute_class_weight('balanced', classes=classes, y=y_train_cl)
pos_weight = cws[1] / (cws[0] + cws[1])
criterion = FocalLoss(alpha_weight=pos_weight, gamma=1.5)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_probs_folds = np.zeros(len(X_test))
val_probs_folds = np.zeros(len(X_val))

best_fold_model_state = None; best_fold_f1 = -1

for fold, (trn_idx, vld_idx) in enumerate(skf.split(X_train_cl, y_train_cl)):
    print(f"--- FOLD {fold+1}/5 ---")
    class_counts = np.bincount(y_train_cl[trn_idx], minlength=2)
    sample_weights = np.array([1.0 / class_counts[y] for y in y_train_cl[trn_idx]])
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
    train_dl = DataLoader(TktDataset(X_train_cl[trn_idx], y_train_cl[trn_idx]), batch_size=16, sampler=sampler)
    valid_dl = DataLoader(TktDataset(X_train_cl[vld_idx], y_train_cl[vld_idx]), batch_size=32)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, hidden_dropout_prob=0.15, attention_probs_dropout_prob=0.15).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.03)
    num_epochs = 4
    scheduler = get_linear_schedule_with_warmup(optimizer, int(len(train_dl)*num_epochs*0.1), len(train_dl)*num_epochs)
    
    for epoch in range(1, num_epochs + 1):
        model.train()
        for b in tqdm(train_dl, desc=f'Fold {fold+1} Ep {epoch}', leave=False):
            optimizer.zero_grad()
            out = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE))
            loss = criterion(out.logits, b['labels'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step()
            
    model.eval()
    fold_vld_logits = []
    with torch.no_grad():
        for b in valid_dl:
            out = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE))
            fold_vld_logits.extend(out.logits.cpu().numpy())
    
    probs = torch.softmax(torch.tensor(fold_vld_logits), dim=1)[:, 1].numpy()
    fold_best = f1_score(y_train_cl[vld_idx], (probs >= 0.5).astype(int), average='macro', zero_division=0)
    for th in np.arange(0.20, 0.81, 0.01):
        pred = (probs >= th).astype(int)
        mf = f1_score(y_train_cl[vld_idx], pred, average='macro', zero_division=0)
        fold_best = max(fold_best, mf)
    if fold_best > best_fold_f1:
        best_fold_f1 = fold_best
        best_fold_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    val_dl = DataLoader(TktDataset(X_val, y_val), batch_size=32)
    fold_val_logits = []
    with torch.no_grad():
        for b in val_dl:
            out = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE))
            fold_val_logits.extend(out.logits.cpu().numpy())
    val_probs_folds += torch.softmax(torch.tensor(fold_val_logits), dim=1)[:, 1].numpy() / 5
    
    test_dl = DataLoader(TktDataset(X_test, y_test), batch_size=32)
    fold_test_logits = []
    with torch.no_grad():
        for b in test_dl:
            out = model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE))
            fold_test_logits.extend(out.logits.cpu().numpy())
    test_probs_folds += torch.softmax(torch.tensor(fold_test_logits), dim=1)[:, 1].numpy() / 5
    
    del model, optimizer; gc.collect(); torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
model.load_state_dict(best_fold_model_state)


## Phase 3: Calibration & Thresholding


In [ ]:
# Leak-free engineered classifier. It may use observable inference-time fields, but not pseudo-label
# answer columns such as inferred_severity_ordinal, severity_delta, fused_severity_score, or sig_*.
LEAKAGE_COLUMNS = [
    'inferred_severity_ordinal', 'severity_delta', 'fused_severity_score',
    'sig_a_severity', 'sig_b_rt_proxy', 'sig_c_rules', 'sig_d_cluster', 'signal_agreement',
    'mismatch_label'
]
IGNORED_ANTI_OVERFIT_COLUMNS = ['Ticket_ID', 'Customer_Name', 'Assigned_Agent', 'Satisfaction_Score', 'Submission_Date']
print('Leakage columns excluded from classifier features:', LEAKAGE_COLUMNS)
print('High-cardinality/post-hoc columns ignored:', [c for c in IGNORED_ANTI_OVERFIT_COLUMNS if c in df.columns])

def build_meta_clf_features(df_split, is_train=False):
    text = (
        df_split['combined_text'].fillna('') +
        ' assigned_priority=' + df_split['Priority_Level'].astype(str) +
        ' channel=' + df_split['Ticket_Channel'].astype(str) +
        ' category=' + df_split['Issue_Category'].astype(str) +
        ' domain=' + df_split['Customer_Domain'].astype(str)
    )
    if is_train:
        build_meta_clf_features.vec = TfidfVectorizer(
            max_features=7000, ngram_range=(1,2), min_df=3, max_df=0.92, sublinear_tf=True
        )
        txt = build_meta_clf_features.vec.fit_transform(text)
        build_meta_clf_features.cat_cols = None
    else:
        txt = build_meta_clf_features.vec.transform(text)

    cats = pd.get_dummies(df_split[['Priority_Level','Ticket_Channel','Issue_Category','RT_Bucket']].astype(str))
    if is_train:
        build_meta_clf_features.cat_cols = cats.columns
    cats = cats.reindex(columns=build_meta_clf_features.cat_cols, fill_value=0)
    nums = df_split[['Resolution_Time_Hours', 'assigned_ordinal']].astype(float).values
    return sparse.hstack([txt, sparse.csr_matrix(cats.values), sparse.csr_matrix(nums)], format='csr')

X_meta_tr = build_meta_clf_features(df_train_clean, True)
X_meta_va = build_meta_clf_features(df_val, False)
X_meta_te = build_meta_clf_features(df_test, False)

scale_pos_weight = (len(y_train_cl) - y_train_cl.sum()) / max(y_train_cl.sum(), 1)
meta_clf = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=2,
    min_child_weight=8,
    learning_rate=0.035,
    subsample=0.80,
    colsample_bytree=0.75,
    reg_alpha=1.0,
    reg_lambda=8.0,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)
meta_clf.fit(X_meta_tr, y_train_cl)
meta_val_probs = meta_clf.predict_proba(X_meta_va)[:, 1]
meta_test_probs = meta_clf.predict_proba(X_meta_te)[:, 1]

# Calibrate DeBERTa probabilities, then tune the ensemble blend and decision threshold on validation only.
calibrator = LogisticRegression(class_weight='balanced')
calibrator.fit(val_probs_folds.reshape(-1, 1), y_val)
val_probs_cal = calibrator.predict_proba(val_probs_folds.reshape(-1, 1))[:, 1]
test_probs_cal = calibrator.predict_proba(test_probs_folds.reshape(-1, 1))[:, 1]

best = {'score': -1, 'blend': 0.5, 'threshold': 0.5, 'metrics': None}
thresh_records = []
for blend in np.arange(0.0, 1.01, 0.05):
    val_blend = blend * val_probs_cal + (1 - blend) * meta_val_probs
    for th in np.arange(0.10, 0.91, 0.01):
        p = (val_blend >= th).astype(int)
        f = f1_score(y_val, p, average='macro', zero_division=0)
        acc = accuracy_score(y_val, p)
        r0 = recall_score(y_val, p, pos_label=0, zero_division=0)
        r1 = recall_score(y_val, p, pos_label=1, zero_division=0)
        meets = (acc >= 0.83 and f >= 0.82 and r0 >= 0.78 and r1 >= 0.78)
        score = f + 0.05 * min(r0, r1) + (0.05 if meets else 0)
        thresh_records.append({'Blend_DeBERTa': blend, 'Threshold': th, 'Accuracy': acc, 'Macro F1': f, 'Recall 0': r0, 'Recall 1': r1})
        if score > best['score']:
            best = {'score': score, 'blend': blend, 'threshold': th, 'metrics': (acc, f, r0, r1)}

best_blend = best['blend']; best_th = best['threshold']
val_probs_final = best_blend * val_probs_cal + (1 - best_blend) * meta_val_probs
test_probs_final = best_blend * test_probs_cal + (1 - best_blend) * meta_test_probs
test_probs_cal = test_probs_final  # keep downstream dossier confidence code compatible
test_preds = (test_probs_final >= best_th).astype(int)

acc = accuracy_score(y_test, test_preds)
mac_f1 = f1_score(y_test, test_preds, average='macro', zero_division=0)
rec = recall_score(y_test, test_preds, average=None, zero_division=0)
if len(rec) < 2:
    rec = np.pad(rec, (0, 2-len(rec)), constant_values=0)

print(f'Best validation blend: {best_blend:.2f} DeBERTa / {1-best_blend:.2f} engineered classifier')
print(f'Best validation threshold: {best_th:.2f}')
print(f'Final Blind Test Accuracy : {acc:.4f} (Target >= 0.83)')
print(f'Final Blind Test Macro F1 : {mac_f1:.4f} (Target >= 0.82)')
print(f'Final Blind Test Recall C0: {rec[0]:.4f} (Target >= 0.78)')
print(f'Final Blind Test Recall C1: {rec[1]:.4f} (Target >= 0.78)')
print('\nClassification report:')
print(classification_report(y_test, test_preds, target_names=['Consistent','Mismatch'], zero_division=0))


## Visual Analytics 3: Classifier Verification (PR, ROC, & Confusion Matrix)


In [ ]:
# Threshold Verification Table
from tabulate import tabulate
df_thresh = pd.DataFrame(thresh_records).sort_values('Macro F1', ascending=False).head(15).round(4)
print("Top Validation Threshold/Blend Candidates:")
print(tabulate(df_thresh, headers='keys', tablefmt='psql'))
print(f"\nOptimal Selected Blend: {best_blend:.2f} DeBERTa / {1-best_blend:.2f} engineered")
print(f"Optimal Selected Threshold: {best_th:.2f}\n")

# Plot PR and ROC on Validation Set
precision, recall, _ = precision_recall_curve(y_val, val_probs_final)
fpr, tpr, _ = roc_curve(y_val, val_probs_final)
roc_auc = auc(fpr, tpr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(recall, precision, color='blue', lw=2)
ax1.set_xlabel('Recall'); ax1.set_ylabel('Precision'); ax1.set_title('Precision-Recall Curve (Validation)')
ax2.plot(fpr, tpr, color='red', lw=2, label=f'AUC = {roc_auc:.3f}')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate'); ax2.set_title('ROC Curve (Validation)')
ax2.legend(loc="lower right")
plt.show()

# Final Test Visuals
fig, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, display_labels=['Consistent', 'Mismatch'], cmap='Blues', ax=ax3)
ax3.set_title("Test Set Confusion Matrix")

sns.barplot(x=['Consistent (Class 0)', 'Mismatch (Class 1)'], y=[rec[0], rec[1]], ax=ax4, palette='pastel')
ax4.axhline(0.78, color='red', linestyle='--', label='Target Minimum (0.78)')
ax4.set_ylim(0, 1.0)
ax4.set_title("Per-Class Recall Verification (Test Set)")
ax4.legend()
plt.show()


## Phase 4: Strict JSON Dossier Extraction


In [ ]:
SEVERITY_INV = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Critical'}

dossiers = []
for i in range(len(df_test)):
    if test_preds[i] == 1:
        row = df_test.iloc[i]
        ass_ord = int(row['assigned_ordinal'])
        inf_ord = int(row['inferred_severity_ordinal'])
        delta = inf_ord - ass_ord
        if delta == 0:
            # Classifier may flag a borderline case; dossier schema requires a calibrated non-zero mismatch.
            continue
        m_type = "Hidden Crisis" if delta > 0 else "False Alarm"
        delta_str = f"+{delta}" if delta > 0 else str(delta)
        rt_val = float(row['Resolution_Time_Hours'])
        text_l = row['combined_text'].lower()
        kws = [k for k in critical_kws + high_kws + esc_kws + low_kws + neg_kws if k in text_l]
        
        feature_evidence = [
            {"signal": "assigned_priority", "value": str(row['Priority_Level']), "weight": "comparison baseline"},
            {"signal": "self_supervised_severity", "value": SEVERITY_INV[inf_ord], "weight": f"fusion={row['fused_severity_score']:.2f}"},
            {"signal": "resolution_time", "value": f"{rt_val:.1f} hours", "interpretation": f"Input field Resolution_Time_Hours falls in {row['RT_Bucket']}."},
            {"signal": "channel", "value": str(row['Ticket_Channel']), "weight": "metadata feature"},
        ]
        if kws:
            feature_evidence.append({"signal": "keyword", "value": ", ".join(sorted(set(kws))[:6]), "weight": "rule feature"})
        
        dossier = {
            "ticket_id": str(row['Ticket_ID']),
            "assigned_priority": SEVERITY_INV[ass_ord],
            "inferred_severity": SEVERITY_INV[inf_ord],
            "mismatch_type": m_type,
            "severity_delta": delta_str,
            "feature_evidence": feature_evidence,
            "constraint_analysis": (
                f"The ticket subject is '{row['Ticket_Subject']}' and the assigned priority is {SEVERITY_INV[ass_ord]}. "
                f"The self-supervised signal fusion infers {SEVERITY_INV[inf_ord]} from the ticket text, channel, category, and recorded resolution time, producing a severity delta of {delta_str}."
            ),
            "confidence": round(float(test_probs_cal[i]), 4)
        }
        dossiers.append(dossier)

print(json.dumps(dossiers[:2], indent=2))
with open('outputs/sia_dossiers.json', 'w') as f:
    json.dump(dossiers, f, indent=2)
